# Day 22 — Project: Sales Order Backlog Analysis
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Replicate SAP VA05/VBAP-style open order analysis using pandas
2. Segment backlog by age, sales org, and distribution channel
3. Identify high-risk orders and build a week-over-week backlog trend

---
> **SAP Context:** VA05 lists open sales orders; VBAP contains line-item details. Key fields: VBELN (order number), ERDAT (creation date), NETWR (net value), VKORG (sales org), VTWEG (distribution channel). A backlog analysis is a core deliverable for S&OP (Sales & Operations Planning) reviews.


## 0. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
TODAY = pd.Timestamp('2026-04-01')

sales = pd.read_csv(DATA / 'sales_orders.csv', parse_dates=['ERDAT'])
print(sales['STATUS'].value_counts())
sales.head(3)

## 1. Filter to OPEN Orders and Calculate Order Age

In [ ]:
# YOUR SOLUTION
# OPEN orders: STATUS == 'OPEN' (inspect actual statuses above)
open_orders = sales[sales['STATUS'] == 'OPEN'].copy()

# Order age in days
open_orders['ORDER_AGE_DAYS'] = (TODAY - open_orders['ERDAT']).dt.days

print(f'Open orders: {len(open_orders)}')
print(f'Total backlog value: ${open_orders["NETWR"].sum():,.0f}')
open_orders[['VBELN','ERDAT','ORDER_AGE_DAYS','NETWR','VKORG','VTWEG']].head(8)

## 2. Segment by Age Bucket, Sales Org, and Distribution Channel

In [ ]:
# YOUR SOLUTION
# Age buckets: Fresh (<14d), Aging (14-30d), At Risk (31-60d), Critical (>60d)
bins = [-1, 13, 30, 60, 9999]
labels = ['Fresh (<14d)', 'Aging (14-30d)', 'At Risk (31-60d)', 'Critical (>60d)']
open_orders['AGE_BUCKET'] = pd.cut(open_orders['ORDER_AGE_DAYS'], bins=bins, labels=labels)

# Summary by segment dimensions
backlog_by_vkorg = (
    open_orders.groupby(['VKORG', 'AGE_BUCKET'], observed=True)
    .agg(ORDER_COUNT=('VBELN', 'count'), TOTAL_VALUE=('NETWR', 'sum'), AVG_AGE=('ORDER_AGE_DAYS', 'mean'))
    .reset_index()
)

backlog_by_vtweg = (
    open_orders.groupby(['VTWEG', 'AGE_BUCKET'], observed=True)
    .agg(ORDER_COUNT=('VBELN', 'count'), TOTAL_VALUE=('NETWR', 'sum'))
    .reset_index()
)

print('=== Backlog by Sales Org + Age Bucket ===')
print(backlog_by_vkorg.to_string(index=False))

## 3. Compute KPI Summary

In [ ]:
# YOUR SOLUTION
kpi = {
    'Total Backlog Value': f"${open_orders['NETWR'].sum():,.0f}",
    'Open Order Count': len(open_orders),
    'Avg Days Open': f"{open_orders['ORDER_AGE_DAYS'].mean():.1f}",
    'Median Days Open': f"{open_orders['ORDER_AGE_DAYS'].median():.1f}",
    'Orders >60 Days Old': (open_orders['ORDER_AGE_DAYS'] > 60).sum(),
    'Value in Critical Bucket': f"${open_orders.loc[open_orders['AGE_BUCKET']=='Critical (>60d)','NETWR'].sum():,.0f}",
}

print('=== Backlog KPI Summary ===')
for k, v in kpi.items():
    print(f'  {k:<30} {v}')

## 4. Flag High-Risk Orders (age >60d AND value >$5k)

In [ ]:
# YOUR SOLUTION
high_risk = open_orders[
    (open_orders['ORDER_AGE_DAYS'] > 60) & (open_orders['NETWR'] > 5000)
].copy()
high_risk = high_risk.sort_values('NETWR', ascending=False)

print(f'High-risk orders: {len(high_risk)}')
print(f'High-risk total value: ${high_risk["NETWR"].sum():,.0f}')
high_risk[['VBELN','KUNNR','MATNR','ERDAT','ORDER_AGE_DAYS','NETWR','VKORG','VTWEG']].head(10)

## 5. Week-over-Week Backlog Trend (simulated from creation dates)

In [ ]:
# YOUR SOLUTION
# Simulate "what was open as of each Monday for the past 12 weeks"
# An order is 'open' on a given week-end if ERDAT <= week_end AND STATUS == 'OPEN'
# (Simplification: we treat all current OPEN orders as still-open going back to their creation)

reference_mondays = pd.date_range(end=TODAY, periods=12, freq='W-MON')

weekly_backlog = []
for wk in reference_mondays:
    # Orders created on or before this week
    snapshot = open_orders[open_orders['ERDAT'] <= wk]
    weekly_backlog.append({
        'WEEK': wk,
        'ORDER_COUNT': len(snapshot),
        'TOTAL_VALUE': snapshot['NETWR'].sum()
    })

wk_df = pd.DataFrame(weekly_backlog)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(wk_df['WEEK'], wk_df['TOTAL_VALUE'] / 1e3, width=5,
        color='#3498db', alpha=0.6, label='Backlog Value ($K)')
ax2.plot(wk_df['WEEK'], wk_df['ORDER_COUNT'], color='#e74c3c',
         marker='o', linewidth=2, label='Order Count')

ax1.set_ylabel('Backlog Value ($K)', color='#3498db')
ax2.set_ylabel('Order Count', color='#e74c3c')
ax1.set_title('Weekly Backlog Trend (Past 12 Weeks)', fontweight='bold')
ax1.tick_params(axis='x', rotation=30)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}K'))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('day22_backlog_trend.png', dpi=120, bbox_inches='tight')
plt.show()

## Daily Prompt
**Answer independently:**

1. In SAP VA05, you can filter by sales org, plant, order type. How would you replicate multi-filter logic in pandas? Write a reusable `filter_open_orders(df, vkorg=None, vtweg=None, min_age=None)` function.
2. Your trend analysis shows backlog grew 40% over 8 weeks. What 3 root-cause hypotheses would you bring to the S&OP meeting, and what data would confirm each?
3. The distribution channel field (`VTWEG`) has values 10, 20, 30. Map them to readable labels (10=Direct, 20=Distributor, 30=Online) and re-run the segment analysis.
